# Data Extraction & Web Scraping — Hands-On Exercise

Welcome! This notebook teaches you how to collect data from the internet using three essential techniques:

1. **REST API** — request structured data from a server using standard HTTP calls
2. **GraphQL** — ask for exactly the fields you need, nothing more
3. **Web Scraping** — extract data directly from HTML when no API exists

**By the end of this notebook you will be able to:**
- Authenticate and paginate through a real REST API (GitHub)
- Write GraphQL queries to fetch precise data
- Parse HTML with BeautifulSoup and handle multi-page scrapes

> **Who this is for:** You know basic Python (variables, loops, functions). You've never called an API or scraped a website before. No worries — everything is explained step by step.

## What You'll Build

By the end of this notebook you'll have extracted five real datasets:

| # | Dataset | How | Output |
|---|---------|-----|--------|
| 1 | `pandas` library release history | REST API | `pandas_releases` DataFrame + `pandas_releases` DuckDB table |
| 2 | Top contributors by commit count | REST API | `contributors` DataFrame + `pandas_contributors` DuckDB table |
| 3 | GraphQL query of `pandas` releases | GraphQL | `graphql_releases` DataFrame |
| 4 | Countries of the world (name, capital, population, area) | Web scraping | `countries_df` DataFrame |
| 5 | NHL hockey team stats (all 24 pages) | Web scraping + pagination | `hockey_stats` DataFrame |

Here's a sneak peek at what `pandas_releases` will look like:

```
   version          published_at                          summary
0  v2.2.1   2024-03-18 15:30:00+00:00  ## What's new in 2.2.1...
1  v2.2.0   2024-01-19 17:00:00+00:00  ## What's new in 2.2.0...
2  v2.1.4   2024-01-08 20:00:00+00:00  ## What's new in 2.1.4...
...
```

---
## Setup

Before we write any code, let's get our tools ready.

### Libraries we'll use

| Library | What it does |
|---------|-------------|
| `requests` | Makes HTTP calls to APIs and websites |
| `python-dotenv` | Loads secret keys from a `.env` file so they stay out of your code |
| `pandas` | Creates DataFrames — like spreadsheets in Python |
| `sqlalchemy` | Lets pandas talk to databases |
| `duckdb` | A fast, file-based database — we'll save our results here |
| `beautifulsoup4` | Parses HTML so we can extract data from web pages |

We will use 'bde' environment for this notebook, which has all these libraries pre-installed. If you're using a different environment, make sure to

Install them if needed:
```bash
pip install requests python-dotenv pandas sqlalchemy duckdb duckdb-engine beautifulsoup4
```

### Step 1 of 2: Get a GitHub Personal Access Token

We need a token so GitHub knows who we are and lets us call their API.

**Follow these steps:**

1. Go to [github.com](https://github.com) and sign in
2. Click your profile photo (top-right) → **Settings**
3. Scroll down the left sidebar → click **Developer settings** (at the very bottom)
4. Click **Personal access tokens** → **Fine-grained tokens** → **Generate new token**
5. Give it a name (e.g. `self-study-scraping`)
6. Set **Expiration** to 30 days
7. Under **Repository access**, select **Public repositories (read-only)**
8. Scroll to the bottom → click **Generate token**
9. **Copy the token immediately** — GitHub only shows it once!

> ⚠️ Keep your token private. Never paste it directly into a notebook or commit it to GitHub.

In [1]:
import os

# Run this cell ONCE to create your .env file.
# After running, open the .env file and paste your token in place of the placeholder.

env_path = '../.env'

if os.path.exists(env_path):
    print(f"ℹ️  .env file already exists at '{env_path}'.")
    print("   Open it and make sure GITHUB_TOKEN is set to your actual token.")
else:
    with open(env_path, 'w') as f:
        f.write("# GitHub personal access token\n")
        f.write("GITHUB_TOKEN='<PASTE-YOUR-TOKEN-HERE>'\n")
    print(f"✅ Created '{env_path}'")
    print("   Now open the file and replace <PASTE-YOUR-TOKEN-HERE> with your actual token.")

ℹ️  .env file already exists at '../.env'.
   Open it and make sure GITHUB_TOKEN is set to your actual token.


### Step 2 of 2: Add your token to the .env file

1. Find the `.env` file in the **root folder** of this project (one level above `notebooks/`)
2. Open it in a text editor
3. Replace `<PASTE-YOUR-TOKEN-HERE>` with your actual token, like this:

```
GITHUB_TOKEN='github_xxxxxxxxxxxxxxxxxxxx'
```

4. Save and close the file

> 📁 The `.env` file is listed in `.gitignore` so it will **never** be pushed to GitHub.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # 👈 reads the .env file and loads GITHUB_TOKEN into the environment
access_token = os.getenv('GITHUB_TOKEN')

# Verify the token loaded correctly (prints first/last 4 chars only — never print the full token!)
if access_token and access_token != '<PASTE-YOUR-TOKEN-HERE>':
    if len(access_token) >= 8:
        print(f"✅ Token loaded: {access_token[:4]}...{access_token[-4:]}")
    else:
        print("⚠️  Token looks too short — double-check you copied it in full.")
else:
    print("❌ Token not found or still a placeholder. Check your .env file.")

✅ Token loaded: gith...BSM2


---
## Chapter 1 — REST API

### The Mission

You've joined a team that tracks the health of popular open-source libraries. Your first task: **collect the full release history of the `pandas` library** — every version, when it was published, and what changed.

Pandas hosts all of this on GitHub. GitHub exposes it via a **REST API** — a web service that returns structured data when you send it the right request. Let's learn how.

### Concept: How a REST API Works

When you call an API, your Python script sends an **HTTP request** to a server. The server processes it and sends back an **HTTP response** containing data (usually in JSON format).

<svg viewBox="0 0 620 110" xmlns="http://www.w3.org/2000/svg" style="font-family:sans-serif;font-size:12px;max-width:620px;display:block;margin:16px 0;">
  <rect x="10" y="35" width="120" height="42" rx="6" fill="#dbeafe" stroke="#3b82f6" stroke-width="1.5"/>
  <text x="70" y="57" text-anchor="middle" fill="#1e40af" font-weight="bold">Your Script</text>
  <text x="70" y="71" text-anchor="middle" fill="#1e40af" font-size="10">(requests.get)</text>
  <rect x="490" y="35" width="120" height="42" rx="6" fill="#dcfce7" stroke="#22c55e" stroke-width="1.5"/>
  <text x="550" y="57" text-anchor="middle" fill="#166534" font-weight="bold">GitHub API</text>
  <text x="550" y="71" text-anchor="middle" fill="#166534" font-size="10">api.github.com</text>
  <defs>
    <marker id="arr-blue" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#3b82f6"/></marker>
    <marker id="arr-green" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#22c55e"/></marker>
  </defs>
  <line x1="130" y1="50" x2="488" y2="50" stroke="#3b82f6" stroke-width="1.5" marker-end="url(#arr-blue)"/>
  <text x="310" y="44" text-anchor="middle" fill="#1d4ed8" font-size="11">① GET /repos/pandas-dev/pandas/releases   +   Authorization header</text>
  <line x1="490" y1="63" x2="132" y2="63" stroke="#22c55e" stroke-width="1.5" marker-end="url(#arr-green)"/>
  <text x="310" y="80" text-anchor="middle" fill="#15803d" font-size="11">② 200 OK   +   JSON array of release objects</text>
</svg>

**Key terms:**

| Term | What it means |
|------|---------------|
| **Endpoint** | The URL you call, e.g. `https://api.github.com/repos/pandas-dev/pandas/releases` |
| **HTTP verb** | The type of action — `GET` = read data, `POST` = send data |
| **Header** | Extra info sent with the request — e.g. your identity token |
| **Status code** | A 3-digit number in the response: `200` = success, `4xx` = your error, `5xx` = server error |
| **JSON** | The data format APIs return — looks like a Python dict/list |
| **Bearer token** | Your API key, passed in the `Authorization` header |

### Concept: JSON is just Python dicts and lists

JSON responses from APIs look like this:

```json
[
  {
    'tag_name': 'v2.2.1',
    'published_at': '2024-03-18T15:30:00Z',
    'body': '## What\'s new in 2.2.1 ...'
  },
  {
    'tag_name': 'v2.2.0',
    'published_at': '2024-01-19T17:00:00Z',
    'body': '## What\'s new in 2.2.0 ...'
  }
]
```

- The outer `[...]` means it's a **list**
- Each `{...}` is a **dict** with key-value pairs
- In Python: `response.json()` gives you the list. `response.json()[0]["tag_name"]` gives you `"v2.2.1"`

### Worked Example: Fetching pandas Releases

Let's call the GitHub releases endpoint step by step. We'll explain every line.

In [49]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
access_token = os.getenv('GITHUB_TOKEN')

response = requests.get(
    "https://api.github.com/repos/pandas-dev/pandas/releases",  # 👈 the endpoint
    headers={
        "Accept": "application/vnd.github+json",   # 👈 tells GitHub we want JSON format
        "Authorization": f"Bearer {access_token}", # 👈 proves who we are
        "X-GitHub-Api-Version": "2022-11-28"       # 👈 pins to a stable API version
    }
)

In [50]:
# A status code of 200 means success.
# Print it so we can confirm the request worked before going further.
print(response.status_code)

200


> **Troubleshooting — if you don't see `200`:**
>
> | Status code | Meaning | Fix |
> |-------------|---------|-----|
> | `401` | Unauthorized — token missing or wrong | Check your `.env` file. Print `access_token` to see if it loaded. |
> | `403` | Forbidden — token doesn't have permission | Create a new token with "Public repositories (read-only)" access |
> | `404` | Not found — URL is wrong | Double-check the endpoint URL for typos |
> | `422` | Unprocessable — bad query parameter | Check `per_page` / `page` values |
>
> **Quick debug:** Run `print(access_token)` to see if the token loaded. If it shows `None`, your `.env` file isn't being found.

In [51]:
# .json() converts the response body from raw text into a Python list/dict
releases = response.json()
print(type(releases))  # should be <class 'list'>
print(len(releases))   # default page size is 30

<class 'list'>
30


In [52]:
# Look at the first release — you'll see a lot of fields we don't need!
# We'll learn how to ask for only what we want in Chapter 2 (GraphQL).
releases[0]

{'url': 'https://api.github.com/repos/pandas-dev/pandas/releases/320615205',
 'assets_url': 'https://api.github.com/repos/pandas-dev/pandas/releases/320615205/assets',
 'upload_url': 'https://uploads.github.com/repos/pandas-dev/pandas/releases/320615205/assets{?name,label}',
 'html_url': 'https://github.com/pandas-dev/pandas/releases/tag/v3.0.3',
 'id': 320615205,
 'author': {'login': 'jorisvandenbossche',
  'id': 1020496,
  'node_id': 'MDQ6VXNlcjEwMjA0OTY=',
  'avatar_url': 'https://avatars.githubusercontent.com/u/1020496?v=4',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/jorisvandenbossche',
  'html_url': 'https://github.com/jorisvandenbossche',
  'followers_url': 'https://api.github.com/users/jorisvandenbossche/followers',
  'following_url': 'https://api.github.com/users/jorisvandenbossche/following{/other_user}',
  'gists_url': 'https://api.github.com/users/jorisvandenbossche/gists{/gist_id}',
  'starred_url': 'https://api.github.com/users/jorisvandenbossche/starred{

In [53]:
# We care about three fields: tag_name, published_at, body
print("Version:", releases[0]['tag_name'])
print("Published:", releases[0]['published_at'])

Version: v3.0.3
Published: 2026-05-11T18:23:29Z


### Pagination — Getting More Than 30 Results

By default GitHub returns 30 results per page. `pandas` has 100+ releases, so we need to page through.

Two query parameters control this:
- `per_page=100` — return up to 100 results per page (the maximum)
- `page=2` — get the second page of results

We add them to the URL after a `?`, separated by `&`:
```
/releases?per_page=100&page=1
/releases?per_page=100&page=2
```

In [54]:
# Get page 1 with 100 results per page
response_p1 = requests.get(
    "https://api.github.com/repos/pandas-dev/pandas/releases?per_page=100&page=1",
    headers={
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {access_token}",
        "X-GitHub-Api-Version": "2022-11-28"
    }
)
releases_p1 = response_p1.json()
print(f"Page 1: {len(releases_p1)} releases")
print("Oldest on page 1:", releases_p1[-1]['tag_name'])

Page 1: 100 releases
Oldest on page 1: v0.18.0rc2


In [55]:
# Get page 2 (releases older than what page 1 returned)
# We fetch it here to demonstrate pagination — we'll use page 1's 100 entries for our DataFrame
response_p2 = requests.get(
    "https://api.github.com/repos/pandas-dev/pandas/releases?per_page=100&page=2",
    headers={
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {access_token}",
        "X-GitHub-Api-Version": "2022-11-28"
    }
)
releases_p2 = response_p2.json()
print(f"Page 2: {len(releases_p2)} releases")
if releases_p2:
    print("Oldest on page 2:", releases_p2[-1]['tag_name'])
else:
    print("Page 2 is empty — all releases fit on page 1")

Page 2: 20 releases
Oldest on page 2: v0.13.0


In [57]:
# Get page 2 (releases older than what page 1 returned)
# We fetch it here to demonstrate pagination — we'll use page 1's 100 entries for our DataFrame
response_p3 = requests.get(
    "https://api.github.com/repos/pandas-dev/pandas/releases?per_page=100&page=3",
    headers={
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {access_token}",
        "X-GitHub-Api-Version": "2022-11-28"
    }
)
releases_p3 = response_p3.json()
print(f"Page 3: {len(releases_p3)} releases")
if releases_p3:
    print("Oldest on page 3:", releases_p3[-1]['tag_name'])
else:
    print("Page 3 is empty — all releases fit on page 1")

Page 3: 0 releases
Page 3 is empty — all releases fit on page 1


### Building a DataFrame

We only need three fields from each release. Let's use a list comprehension to extract them, then build a DataFrame.

In [11]:
import pandas as pd

# Extract only the fields we need from page 1 (the most recent 100 releases)
records = [
    {
        "version": r["tag_name"],          # 👈 the version number e.g. "v2.2.1"
        "published_at": r["published_at"], # 👈 ISO 8601 date string
        "summary": r["body"]               # 👈 the release notes
    }
    for r in releases_p1
]
print(records)
pandas_releases = pd.DataFrame(records)
pandas_releases

[{'version': 'v3.0.3', 'published_at': '2026-05-11T18:23:29Z', 'summary': 'We are pleased to announce the release of pandas 3.0.3.\r\nThis is a patch release in the 3.0.x series and includes some regression fixes and bug fixes. We recommend that all users of the 3.0.x series upgrade to this version.\r\n\r\nSee the [full whatsnew](https://pandas.pydata.org/docs/whatsnew/v3.0.3.html) for a list of all the changes.\r\n\r\nPandas 3.0 supports Python 3.11 and higher. \r\nThe release can be installed from PyPI:\r\n\r\n    python -m pip install --upgrade pandas==3.0.*\r\n\r\nOr from conda-forge\r\n\r\n    conda install -c conda-forge pandas=3.0\r\n\r\nPlease report any issues with the release on the [pandas issue tracker](https://github.com/pandas-dev/pandas/issues).\r\n\r\nThanks to all the contributors who made this release possible.'}, {'version': 'v3.0.2', 'published_at': '2026-03-30T20:14:45Z', 'summary': 'We are pleased to announce the release of pandas 3.0.2.\r\nThis is a patch release

,version,published_at,summary
0,v3.0.3,2026-05-11T18:23:29Z,We are pleased to announce the release of pand...
1,v3.0.2,2026-03-30T20:14:45Z,We are pleased to announce the release of pand...
2,v3.0.1,2026-02-17T21:56:03Z,We are pleased to announce the release of pand...
3,v3.0.0,2026-01-21T15:23:43Z,We are pleased to announce the release of pand...
4,v3.0.0rc2,2026-01-14T22:17:15Z,
...,...,...,...
95,v0.19.0,2016-10-02T14:17:50Z,This is a major release from 0.18.1 and includ...
96,v0.19.0rc1,2016-09-07T20:50:22Z,**RELEASE CANDIDATE**\n\nThis is a major relea...
97,v0.18.1,2016-05-03T14:48:48Z,This is a minor release from 0.18.0 and includ...
98,v0.18.0,2016-03-12T15:12:32Z,This is a major release from 0.17.1 and includ...


In [12]:
# Convert published_at from a string to a proper datetime object
# This lets us do date arithmetic (e.g. calculate time between releases)
pandas_releases['published_at'] = pd.to_datetime(pandas_releases['published_at'])
pandas_releases.dtypes

version                      object
published_at    datetime64[ns, UTC]
summary                      object
dtype: object

### Saving to DuckDB

Now let's persist the data to a database. We use **DuckDB** — a lightweight, file-based database that needs no server setup. `SQLAlchemy` gives pandas a way to write DataFrames directly into it using `.to_sql()`.

In [13]:
import sqlalchemy as sqla
import os

# Get the absolute path to the project root (one level above notebooks/)
parent_dir = os.path.abspath(os.path.pardir)

# Ensure the output directory exists
os.makedirs(f'{parent_dir}/output', exist_ok=True)

# Create a connection to the DuckDB file (unit-2-4.db is created fresh for this notebook)
engine = sqla.create_engine(f'duckdb:///{parent_dir}/output/unit-2-4.db')

In [14]:
# Write the DataFrame to a table called "pandas_releases"
# if_exists='replace' will overwrite the table if it already exists
try:
    pandas_releases.to_sql("pandas_releases", engine, if_exists='replace', index=False)
    print("✅ Saved pandas_releases to DuckDB")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Saved pandas_releases to DuckDB


In [15]:
# Always dispose the engine when done — this closes the connection
# You can now open output/unit-2-4.db in DBeaver or any DuckDB client
engine.dispose()
print("Connection closed.")

Connection closed.


---
### ✏️ Exercise 1: Who Has Contributed the Most Commits to pandas?

The `stats/contributors` endpoint returns a list of contributors and their commit counts.

**Your task:**
1. Call `https://api.github.com/repos/pandas-dev/pandas/stats/contributors` with the same headers as before
2. Extract the `login` (contributor username) and `total` (commit count) from each item
3. Build a DataFrame named `contributors` with columns `login` and `total_commits`
4. Find the contributor with the highest number of commits
5. Save the DataFrame to DuckDB as a table named `pandas_contributors`

> ⚠️ **Note on this endpoint:** GitHub computes contributor stats on demand. If you get a `202` status code (instead of `200`), the data isn't ready yet — wait 10 seconds and try again.

---

**💡 Hint 1 — The data structure:**  
Each item in the response list looks like this:
```python
{
    'total': 1234,         # total commits by this contributor
    'author': {
        'login': 'jreback' # their GitHub username
    },
    'weeks': [...]         # weekly breakdown — ignore this
}
```

**💡 Hint 2 — Accessing nested values:**  
To get the login, use `item["author"]["login"]`. To get total commits, use `item["total"]`.

**💡 Hint 3 — Finding the top contributor:**  
After building the DataFrame, use `.sort_values("total_commits", ascending=False).head(1)` to find the contributor with the most commits.

**💡 Hint 4 — Saving to DuckDB:**  
The engine was closed after the releases save. Recreate it before saving:
```python
engine2 = sqla.create_engine(f'duckdb:///{parent_dir}/output/unit-2-4.db')
contributors.to_sql("pandas_contributors", engine2, if_exists='replace', index=False)
engine2.dispose()
```

In [ ]:
# Your code here
# step 1
import requests
import os
from dotenv import load_dotenv

load_dotenv()
access_token = os.getenv('GITHUB_TOKEN')

response = requests.get(
    "https://api.github.com/repos/pandas-dev/pandas/stats/contributors?per_page=80&page=1",  # 👈 the endpoint
    headers={
        "Accept": "application/vnd.github+json",   # 👈 tells GitHub we want JSON format
        "Authorization": f"Bearer {access_token}", # 👈 proves who we are
        "X-GitHub-Api-Version": "2022-11-28"       # 👈 pins to a stable API version
    }
)

print(response.status_code)
contributors = response.json()
print(type(contributors)) 
print(len(contributors))   
contributors[0]

200
<class 'list'>
100


{'total': 21,
 'weeks': [{'w': 1248566400, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1249171200, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1249776000, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1250380800, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1250985600, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1251590400, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1252195200, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1252800000, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1253404800, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1254009600, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1254614400, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1255219200, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1255824000, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1256428800, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1257033600, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1257638400, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1258243200, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1258848000, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1259452800, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1260057600, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1260662400, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1261267200, 'a': 0, 'd':

In [ ]:
# Step 2
import pandas as pd

# Extract only the fields we need from page 1 (the most recent 100 releases)
records = [
    {
        "total": r["total"],          
        "login": r["author"]["login"]          
    }
    for r in contributors
]
print(records)
pandas_contributors = pd.DataFrame(records)
pandas_contributors

[{'total': 21, 'login': 'ryankarlos'}, {'total': 21, 'login': 'noatamir'}, {'total': 21, 'login': 'moink'}, {'total': 22, 'login': 'ganevgv'}, {'total': 22, 'login': 'minggli'}, {'total': 22, 'login': 'Licht-T'}, {'total': 22, 'login': 'afeld'}, {'total': 23, 'login': 'FHaase'}, {'total': 23, 'login': 'danielballan'}, {'total': 25, 'login': 'GYHHAHA'}, {'total': 25, 'login': 'erfannariman'}, {'total': 25, 'login': 'thoo'}, {'total': 26, 'login': 'rsm-23'}, {'total': 26, 'login': 'JustinZhengBC'}, {'total': 26, 'login': 'realead'}, {'total': 26, 'login': 'stephenwlin'}, {'total': 27, 'login': 'davidastephens'}, {'total': 29, 'login': 'Charlie-XIAO'}, {'total': 29, 'login': 'weikhor'}, {'total': 29, 'login': 'evanpw'}, {'total': 29, 'login': 'immerrr'}, {'total': 29, 'login': 'yarikoptic'}, {'total': 30, 'login': 'wjandrea'}, {'total': 30, 'login': 'snitish'}, {'total': 30, 'login': 'debnathshoham'}, {'total': 30, 'login': 'rockg'}, {'total': 30, 'login': 'gliptak'}, {'total': 31, 'login

,total,login
0,21,ryankarlos
1,21,noatamir
2,21,moink
3,22,ganevgv
4,22,minggli
...,...,...
95,1735,jorisvandenbossche
96,1952,mroeschke
97,3129,wesm
98,4794,jreback


In [84]:
# Step 3 (Method 1)
max_contributor = pandas_contributors.loc[pandas_contributors['total'].idxmax()] # return a Series
print(f"The contributor with the most commits is: {max_contributor['login']}")

The contributor with the most commits is: jbrockmendel


In [85]:
# Step 3 (Method 2)
max_contributor = pandas_contributors.sort_values("total", ascending=False).head(1) # return a DataFrame
print(f"The contributor with the most commits is: {max_contributor.iloc[0,1]}")

The contributor with the most commits is: jbrockmendel


In [86]:
# Step 4
import sqlalchemy as sqla
import os

parent_dir = os.path.abspath(os.path.pardir)
os.makedirs(f'{parent_dir}/output', exist_ok=True)
engine1 = sqla.create_engine(f'duckdb:///{parent_dir}/output/unit-2-4-Ex1.db')

try:
    pandas_contributors.to_sql("pandas_contributors", engine1, if_exists='replace', index=False)
    print("✅ Saved pandas_contributors to DuckDB")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Saved pandas_contributors to DuckDB


In [87]:
engine1.dispose()
print("Connection closed.")

Connection closed.


---
## Chapter 2 — GraphQL

### The Mission

You got the releases data from REST — but the response contained dozens of fields you didn't need (`assets`, `author`, `reactions`, etc.). Each wasted field costs bandwidth and processing time. 

Your team asks: **"Can we fetch only the exact fields we need?"**

Enter **GraphQL** — GitHub's newer API that lets you specify exactly what you want. Same data, but you're in control of the shape.

### Concept: REST vs GraphQL

**The over-fetching problem with REST:**

```
REST response for one release (~30 fields):
{
  'url': '...',           ← don't need
  'assets_url': '...',    ← don't need
  'upload_url': '...',    ← don't need
  'html_url': '...',      ← don't need
  'id': 12345,            ← don't need
  'author': {...},        ← don't need
  'tag_name': 'v2.2.1',  ✅ need
  'published_at': '...', ✅ need
  'body': '...',         ✅ need
  ... 21 more fields ...  ← don't need
}
```

**With GraphQL, you ask for only what you need:**

```graphql
query {
  repository(owner: "pandas-dev", name: "pandas") {
    releases(first: 100) {
      edges {
        node {
          tagName       ← only this
          publishedAt   ← only this
          description   ← only this
        }
      }
    }
  }
}
```

<svg viewBox="0 0 620 90" xmlns="http://www.w3.org/2000/svg" style="font-family:sans-serif;font-size:11px;max-width:620px;display:block;margin:16px 0;">
  <rect x="10" y="20" width="120" height="50" rx="5" fill="#dbeafe" stroke="#3b82f6" stroke-width="1.5"/>
  <text x="70" y="43" text-anchor="middle" fill="#1e40af" font-weight="bold">Your Script</text>
  <text x="70" y="58" text-anchor="middle" fill="#1e40af" font-size="10">requests.post</text>
  <rect x="490" y="20" width="120" height="50" rx="5" fill="#f3e8ff" stroke="#9333ea" stroke-width="1.5"/>
  <text x="550" y="43" text-anchor="middle" fill="#6b21a8" font-weight="bold">GraphQL API</text>
  <text x="550" y="58" text-anchor="middle" fill="#6b21a8" font-size="10">api.github.com/graphql</text>
  <defs>
    <marker id="arr-p" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#9333ea"/></marker>
    <marker id="arr-pb" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#7c3aed"/></marker>
  </defs>
  <line x1="130" y1="37" x2="488" y2="37" stroke="#9333ea" stroke-width="1.5" marker-end="url(#arr-p)"/>
  <text x="310" y="31" text-anchor="middle" fill="#7e22ce" font-size="10">POST /graphql  +  JSON body: {"query": "..."}</text>
  <line x1="490" y1="50" x2="132" y2="50" stroke="#7c3aed" stroke-width="1.5" marker-end="url(#arr-pb)"/>
  <text x="310" y="67" text-anchor="middle" fill="#6b21a8" font-size="10">200 OK  +  {"data": { exactly what you asked for }}</text>
</svg>

**Key differences:**

| | REST | GraphQL |
|--|------|---------|
| Endpoint | One per resource (`/releases`, `/issues`, ...) | Single endpoint (`/graphql`) |
| HTTP verb | `GET` for reading | `POST` (always) |
| Response shape | Fixed — server decides | Flexible — you decide |
| Over-fetching | Common | Never |

### Concept: Anatomy of a GraphQL Query

```graphql
query {                                          # ← operation type: "query" means read
  repository(owner: "pandas-dev", name: "pandas") {  # ← the object to start from, with arguments
    releases(first: 100) {                       # ← sub-field with argument (limit to 100)
      totalCount                                 # ← a scalar field (just returns a number)
      edges {                                    # ← "edges" contains the items in the list
        node {                                   # ← "node" is the actual release object
          tagName                                # ← scalar fields we want on each release
          description
          publishedAt
        }
      }
    }
  }
}
```

- **`edges` and `node`** — GraphQL uses a "connection" pattern for lists. `edges` is the list, each `node` is one item.
- **`totalCount`** — a bonus: we get the total count for free without fetching all records.
- **Nested fields** — you can go as deep as the schema allows. Each `{...}` block specifies which sub-fields to return.

### Worked Example: Fetching Releases via GraphQL

Let's replicate what we did with REST — but using a GraphQL query. Notice that we use `POST` (not `GET`) and send the query in the request body.

In [21]:
import requests
import os
from dotenv import load_dotenv
import pandas as pd
from pprint import pprint

load_dotenv()
access_token = os.getenv('GITHUB_TOKEN')

# Define the query as a multi-line string
query = """
query {
    repository(owner: "pandas-dev", name: "pandas") {
        releases(first: 100) {
            totalCount
            edges {
                node {
                    tagName
                    description
                    publishedAt
                }
            }
        }
    }
}
"""

# GraphQL always uses POST, and the query goes in the JSON body
gql_response = requests.post(
    "https://api.github.com/graphql",           # 👈 single endpoint for all GraphQL queries
    headers={"Authorization": f"Bearer {access_token}"},
    json={"query": query}                        # 👈 query is sent as a JSON field called "query"
)

print(gql_response.status_code)  # should be 200

# > GraphQL always returns HTTP 200 even for errors.
# > If the next cell crashes, print gql_data and look for an "errors" key.
gql_data = gql_response.json()
pprint(gql_data)

200
{'data': {'repository': {'releases': {'edges': [{'node': {'description': 'We '
                                                                         'are '
                                                                         'pleased '
                                                                         'to '
                                                                         'announce '
                                                                         'the '
                                                                         'release '
                                                                         'of '
                                                                         'pandas '
                                                                         '3.0.3.\r\n'
                                                                         'This '
                                                                         'is a '
               

In [17]:
# GraphQL wraps all results under a "data" key
# Navigate: gql_data → repository → releases → totalCount
print("Total releases:", gql_data['data']['repository']['releases']['totalCount'])

Total releases: 120


In [22]:
# Get the list of releases — remember: edges → node
edges = gql_data['data']['repository']['releases']['edges']
print("Releases returned:", len(edges))

# Look at the first release (Jupyter displays the last expression in a cell automatically)
edges[0]['node']

Releases returned: 100


{'tagName': 'v3.0.3',
 'description': 'We are pleased to announce the release of pandas 3.0.3.\r\nThis is a patch release in the 3.0.x series and includes some regression fixes and bug fixes. We recommend that all users of the 3.0.x series upgrade to this version.\r\n\r\nSee the [full whatsnew](https://pandas.pydata.org/docs/whatsnew/v3.0.3.html) for a list of all the changes.\r\n\r\nPandas 3.0 supports Python 3.11 and higher. \r\nThe release can be installed from PyPI:\r\n\r\n    python -m pip install --upgrade pandas==3.0.*\r\n\r\nOr from conda-forge\r\n\r\n    conda install -c conda-forge pandas=3.0\r\n\r\nPlease report any issues with the release on the [pandas issue tracker](https://github.com/pandas-dev/pandas/issues).\r\n\r\nThanks to all the contributors who made this release possible.',
 'publishedAt': '2026-05-11T18:23:29Z'}

In [23]:
# Build a DataFrame — same approach as REST but navigating the nested structure
graphql_releases = pd.DataFrame([
    {
        "version": edge["node"]["tagName"],
        "published_at": edge["node"]["publishedAt"],
        "description": edge["node"]["description"]
    }
    for edge in edges
])

graphql_releases.head()

,version,published_at,description
0,v3.0.3,2026-05-11T18:23:29Z,We are pleased to announce the release of pand...
1,v3.0.2,2026-03-30T20:14:45Z,We are pleased to announce the release of pand...
2,v3.0.1,2026-02-17T21:56:03Z,We are pleased to announce the release of pand...
3,v3.0.0,2026-01-21T15:23:43Z,We are pleased to announce the release of pand...
4,v3.0.0rc2,2026-01-14T22:17:15Z,


---
### ✏️ Exercise 2: Query the First 50 Issues via GraphQL

Now write your own GraphQL query. Fetch the first 50 open issues from the `pandas` repository with these fields: `title`, `createdAt`, and the author's `login`.

**Your task:**
1. Write a GraphQL query for the `issues` resource (use `first: 50`)
2. On each issue node, request: `title`, `createdAt`, and `author { login }`
3. Send it via `requests.post` to `https://api.github.com/graphql`
4. Navigate the response and build a DataFrame with columns: `title`, `created_at`, `author`

---

**💡 Hint 1 — Query structure:**  
Start with `repository(owner: "pandas-dev", name: "pandas")`, then `issues(first: 50)`, then `edges { node { ... } }`.

**💡 Hint 2 — The `author` field is nested:**  
`author` is an object, not a scalar. Inside it, request `login`:
```graphql
author {
  login
}
```

**💡 Hint 3 — Navigating the response:**  
The path is: `data['data']['repository']['issues']['edges']`  
Each edge's node has: `edge['node']['title']`, `edge['node']['createdAt']`, `edge['node']['author']['login']`

> ⚠️ **Watch out:** `author` can be `null` for deleted accounts. Use this safe pattern:  
> `edge['node']['author']['login'] if edge['node']['author'] else None`

In [ ]:
# Your code here
# Step 1
import requests
import os
from dotenv import load_dotenv
import pandas as pd
from pprint import pprint

load_dotenv()
access_token = os.getenv('GITHUB_TOKEN')

# Define the query as a multi-line string
query = """
query {
    repository(owner: "pandas-dev", name: "pandas") {
        issues(first: 50, states:OPEN) {
            totalCount
            edges {
                node {
                    title
                    createdAt
                    author {
                        login
                    }
                }
            }
        }
    }
}
"""

# GraphQL always uses POST, and the query goes in the JSON body
gql_response = requests.post(
    "https://api.github.com/graphql",           # 👈 single endpoint for all GraphQL queries
    headers={"Authorization": f"Bearer {access_token}"},
    json={"query": query}                        # 👈 query is sent as a JSON field called "query"
)

print(gql_response.status_code)  # should be 200

# > GraphQL always returns HTTP 200 even for errors.
# > If the next cell crashes, print gql_data and look for an "errors" key.
gql_data = gql_response.json()
pprint(gql_data)

200
{'data': {'repository': {'issues': {'edges': [{'node': {'author': {'login': 'sinhrks'},
                                                        'createdAt': '2012-09-16T18:44:46Z',
                                                        'title': 'Using '
                                                                 'pandas.TimeSeries_DateFormatter '
                                                                 'in bar '
                                                                 'plot?'}},
                                              {'node': {'author': {'login': 'metakermit'},
                                                        'createdAt': '2013-01-17T13:26:47Z',
                                                        'title': "closed='both' "
                                                                 'option for '
                                                                 'resample'}},
                                              {'node': {'author': 

In [10]:
# Step 2
edges = gql_data['data']['repository']['issues']['edges']
print("Releases returned:", len(edges))

# Look at the first release (Jupyter displays the last expression in a cell automatically)
edges[0]['node']

Releases returned: 50


{'title': 'Using pandas.TimeSeries_DateFormatter in bar plot?',
 'createdAt': '2012-09-16T18:44:46Z',
 'author': {'login': 'sinhrks'}}

In [25]:
# Step 3
graphql_issues = pd.DataFrame([
    {
        "title": edge["node"]["title"],
        "created_at": edge["node"]["createdAt"],
        "author": edge["node"]["author"]["login"] if edge["node"]["author"] else None
    }
    for edge in edges
])

graphql_issues.head(15)

,title,created_at,author
0,Using pandas.TimeSeries_DateFormatter in bar p...,2012-09-16T18:44:46Z,sinhrks
1,closed='both' option for resample,2013-01-17T13:26:47Z,metakermit
2,"DataFrame.copy(), at least, should be threadsafe",2013-01-23T00:06:26Z,bshanks
3,BUG: min on non-numeric data with nans,2013-07-06T12:37:58Z,hayd
4,ENH/BUG: Rename of MultiIndex DataFrames does ...,2013-07-08T13:52:55Z,aschilling
5,"Make bar plot on secondary axis, the line plot...",2013-07-28T05:43:36Z,tesla1060
6,ER: compound dtypes - DataFrame constructor/as...,2013-08-05T15:04:16Z,mamikonyan
7,ENH: Add decimal parameter to to_numeric,2013-08-26T13:59:28Z,cancan101
8,Accessing unused columns in pivot_table,2013-09-02T09:11:05Z,wabee
9,ER: should constructors check and raise if big...,2013-09-03T20:13:39Z,jreback


---
## Chapter 3 — Web Scraping

### The Mission

Not all data has an API. You need **world country statistics** — population, capital, area — for a geography dashboard. There's no API for this dataset, but the data is sitting right there on a public website.

**Web scraping** lets you extract data directly from a website's HTML. Instead of calling an API, you download the page and parse the HTML to find the data you need.

### Concept: HTML is a Tree of Tags

Every webpage is built from **HTML tags** nested inside each other, forming a tree structure. BeautifulSoup lets you navigate and search this tree.

<svg viewBox="0 0 580 220" xmlns="http://www.w3.org/2000/svg" style="font-family:monospace;font-size:12px;max-width:580px;display:block;margin:16px 0;">
  <!-- html root -->
  <rect x="240" y="5" width="100" height="28" rx="4" fill="#fef9c3" stroke="#ca8a04" stroke-width="1.5"/>
  <text x="290" y="24" text-anchor="middle" fill="#713f12">&lt;html&gt;</text>
  <!-- head -->
  <rect x="100" y="55" width="100" height="28" rx="4" fill="#e0f2fe" stroke="#0284c7" stroke-width="1.5"/>
  <text x="150" y="74" text-anchor="middle" fill="#075985">&lt;head&gt;</text>
  <!-- body -->
  <rect x="380" y="55" width="100" height="28" rx="4" fill="#e0f2fe" stroke="#0284c7" stroke-width="1.5"/>
  <text x="430" y="74" text-anchor="middle" fill="#075985">&lt;body&gt;</text>
  <!-- lines from html -->
  <line x1="290" y1="33" x2="150" y2="55" stroke="#9ca3af" stroke-width="1.5"/>
  <line x1="290" y1="33" x2="430" y2="55" stroke="#9ca3af" stroke-width="1.5"/>
  <!-- title under head -->
  <rect x="60" y="110" width="100" height="28" rx="4" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/>
  <text x="110" y="129" text-anchor="middle" fill="#166534">&lt;title&gt;</text>
  <line x1="150" y1="83" x2="110" y2="110" stroke="#9ca3af" stroke-width="1.5"/>
  <!-- div, p under body -->
  <rect x="290" y="110" width="100" height="28" rx="4" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/>
  <text x="340" y="129" text-anchor="middle" fill="#166534">&lt;div&gt;</text>
  <rect x="430" y="110" width="100" height="28" rx="4" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/>
  <text x="480" y="129" text-anchor="middle" fill="#166534">&lt;p&gt;</text>
  <line x1="430" y1="83" x2="340" y2="110" stroke="#9ca3af" stroke-width="1.5"/>
  <line x1="430" y1="83" x2="480" y2="110" stroke="#9ca3af" stroke-width="1.5"/>
  <!-- h3, span under div -->
  <rect x="240" y="165" width="80" height="28" rx="4" fill="#fce7f3" stroke="#db2777" stroke-width="1.5"/>
  <text x="280" y="184" text-anchor="middle" fill="#831843">&lt;h3&gt;</text>
  <rect x="360" y="165" width="80" height="28" rx="4" fill="#fce7f3" stroke="#db2777" stroke-width="1.5"/>
  <text x="400" y="184" text-anchor="middle" fill="#831843">&lt;span&gt;</text>
  <line x1="340" y1="138" x2="280" y2="165" stroke="#9ca3af" stroke-width="1.5"/>
  <line x1="340" y1="138" x2="400" y2="165" stroke="#9ca3af" stroke-width="1.5"/>
  <!-- annotations -->
  <text x="8" y="174" fill="#6b7280" font-size="10">soup.find('h3')</text>
  <text x="8" y="188" fill="#db2777" font-size="10" font-weight="bold">↗ finds this</text>
</svg>

**BeautifulSoup lets you search by:**
- **Tag name:** `soup.find('h3')` — finds the first `<h3>` tag
- **Tag + class:** `soup.find_all('h3', 'country-name')` — finds all `<h3>` tags with class `country-name`
- **CSS selector:** `soup.select('div.country > h3')` — uses CSS selector syntax

### Concept: CSS Selectors

CSS selectors let you target HTML elements precisely. Here are the ones you'll use most:

| Selector | Example | What it matches |
|----------|---------|-----------------|
| Tag | `h3` | All `<h3>` elements |
| Class | `.country-name` | All elements with `class="country-name"` |
| Tag + class | `h3.country-name` | `<h3>` elements with `class="country-name"` |
| Child | `div > p` | `<p>` that is a direct child of `<div>` |

**In BeautifulSoup:**
```python
soup.find('h3', 'country-name')          # first <h3 class="country-name">
soup.find_all('span', 'country-capital') # all <span class="country-capital">
```

### ⚖️ Ethics & Legality of Web Scraping

Before scraping any site, check:

1. **`robots.txt`** — visit `https://example.com/robots.txt` to see what the site allows bots to access
2. **Terms of Service** — some sites explicitly prohibit scraping
3. **Rate limiting** — always add `time.sleep()` between requests. Rapid requests can overload a server
4. **Copyright** — scraped data may be copyrighted. Check before redistribution

> We're using [scrapethissite.com](https://www.scrapethissite.com) — a site designed specifically for scraping practice. It's safe and legal to scrape.

### Worked Example: Scraping Country Data

We'll scrape [scrapethissite.com/pages/simple/](https://www.scrapethissite.com/pages/simple/) — a page listing all countries with their capital, population, and area.

**Steps:**
1. Download the page with `requests.get()`
2. Parse the HTML with `BeautifulSoup`
3. Find the elements containing the data
4. Extract text and build a DataFrame

In [25]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Download the page
r = requests.get("https://www.scrapethissite.com/pages/simple/")  # 👈 same requests.get as Chapter 1
print(r.status_code)   # 👈 200 = success; anything else means the page didn't load
print(type(r.text))    # 👈 r.text is the raw HTML as a string — this is what we'll parse

200
<class 'str'>


In [26]:
pprint(r.text)

('<!doctype html>\n'
 '<html lang="en">\n'
 '  <head>\n'
 '    <meta charset="utf-8">\n'
 '    <title>Countries of the World: A Simple Example | Scrape This Site | A '
 'public sandbox for learning web scraping</title>\n'
 '    <link rel="icon" type="image/png" href="/static/images/scraper-icon.png" '
 '/>\n'
 '\n'
 '    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n'
 '    <meta name="description" content="A single page that lists information '
 'about all the countries in the world. Good for those just get started with '
 'web scraping.">\n'
 '\n'
 '    <link '
 'href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css" '
 'rel="stylesheet" '
 'integrity="sha256-MfvZlkHCEqatNoGiOXveE8FIwMzZg4W85qfrfIFBfYc= '
 'sha512-dTfge/zgoMYpP7QbHy4gWMEGsbsdZeCXz7irItjcC3sPUFtf0kuFbDz/ixG7ArTxmDjLXDmezHubeNikyKGVyQ==" '
 'crossorigin="anonymous">\n'
 "    <link href='https://fonts.googleapis.com/css?family=Lato:400,700' "
 "rel='stylesheet' type='text/

In [27]:
# Parse the HTML — BeautifulSoup builds the tree structure we can navigate
soup = BeautifulSoup(r.text, "html.parser")
# 👈 "html.parser" is Python's built-in parser — no extra install needed

**Inspect the page first:** Right-click on a country name in your browser → "Inspect Element". You'll see something like:

```html
<div class="col-md-4 country">
  <h3 class="country-name">Andorra</h3>
  <span class="country-capital">Andorra la Vella</span>
  <span class="country-population">84000</span>
  <span class="country-area">468.0</span>
</div>
```

Each country is a `<div class="col-md-4 country">` containing child elements with specific classes. We'll use those classes to target exactly what we need.

In [28]:
# Find all country name elements — these are <h3> tags with class "country-name"
country_name_elements = soup.find_all('h3', 'country-name')  # 👈 tag name first, then class name
print(f"Found {len(country_name_elements)} country names")
country_name_elements[0]   # 👈 see the raw tag — useful to confirm we found the right element

Found 250 country names


<h3 class="country-name">
<i class="flag-icon flag-icon-ad"></i>
                            Andorra
                        </h3>

In [29]:
# Extract just the text — .text gives the raw text content inside the tag
print(country_name_elements[0].text.strip())  # 👈 .strip() removes leading/trailing whitespace

Andorra


> **Troubleshooting — if `find_all()` returns an empty list:**
>
> | Problem | Fix |
> |---------|-----|
> | Wrong class name | Right-click → Inspect Element in your browser and double-check the exact class string — class names are case-sensitive |
> | Page didn't load | Check `r.status_code` is `200`. If not, the site may be down or blocking you |
> | Site structure changed | Websites change their HTML without warning. Re-inspect the page to find the new class names |
>
> **Quick debug:** Print `r.text[:500]` to see the first 500 characters of the HTML. If you see `<html>` and `<body>` tags, the page loaded correctly. If you see an error message, the request failed.

For efficiency, let's collect all data in one loop — for each country `<div>`, extract all four fields at once using `.find()` on the child elements.

In [30]:
countries = []

# Each country is a <div> with two classes: "col-md-4" and "country"
# Pass both as a list: class_=["col-md-4", "country"] — clearer than a space-separated string
for country_div in soup.find_all('div', class_=["col-md-4", "country"]):  # 👈 matches divs that have BOTH classes
    country = {}
    country['name']       = country_div.find('h3').text.strip()                           # 👈 find the <h3> child and grab its text
    country['capital']    = country_div.find('span', 'country-capital').text.strip()       # 👈 find <span class="country-capital">
    country['population'] = country_div.find('span', 'country-population').text.strip()    # 👈 find <span class="country-population">
    country['area_km2']   = country_div.find('span', 'country-area').text.strip()          # 👈 find <span class="country-area">
    countries.append(country)                                                               # 👈 add this country's dict to our list

countries_df = pd.DataFrame(countries)  # 👈 one row per country, four columns
countries_df.head()

,name,capital,population,area_km2
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0


In [32]:
countries = [
    {
        'name':       d.find('h3').text.strip(),
        'capital':    d.find('span', 'country-capital').text.strip(),
        'population': d.find('span', 'country-population').text.strip(),
        'area_km2':   d.find('span', 'country-area').text.strip()
    }
    for d in soup.find_all('div', class_=["col-md-4", "country"])
]
countries_df = pd.DataFrame(countries)  # 👈 one row per country, four columns
countries_df.head()

,name,capital,population,area_km2
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0


### Multi-Page Scraping (Optional)

Some sites split data across multiple pages. When you click "Page 2" the URL changes — for example:

```
https://www.scrapethissite.com/pages/forms/?page_num=2
```

We can automate this with a `for` loop. Here's a helper function that parses one page's table into a list of row dicts — you'll use it in the exercise below.

In [49]:
import time

def parse_hockey_page(html_text: str) -> list:
    """
    Parse one page of the hockey stats table and return a list of row dicts.
    
    Args:
        html_text: Raw HTML string from requests.get().text
    
    Returns:
        List of dicts, one per team row. Returns [] if the table can't be found.
    """
    soup = BeautifulSoup(html_text, "html.parser")
    
    # Target the <thead> row specifically — more reliable than soup.find('tr')
    # which would grab the first <tr> anywhere on the page
    #thead = soup.find('thead')
    #if not thead:
    #    return []  # 👈 guard: if the page structure changes, return empty rather than crash
    #header_row = thead.find('tr')
    # There is no thead in the html, so changed the code as below to make it more robust
    header_row = soup.find('thead').find('tr') if soup.find('thead') else soup.find('tr')
    headers = [th.text.strip() for th in header_row.find_all('th')]
    
    
    rows = []
    for team_row in soup.find_all('tr', 'team'):
        row = {}
        # zip() pairs each header name with the matching <td> cell — stops at the shorter list
        for header, cell in zip(headers, team_row.find_all('td')):  # 👈 zip pairs header to cell
            row[header] = cell.text.strip()
        rows.append(row)
    return rows

---
### ✏️ Exercise 3: Scrape All 24 Pages of NHL Hockey Stats

Use the `parse_hockey_page()` helper above to scrape all 24 pages from [scrapethissite.com/pages/forms/](https://www.scrapethissite.com/pages/forms/).

**Your task:**
1. Loop through pages 1–24 (the URL pattern is `?page_num=N`)
2. For each page, call `requests.get()` then `parse_hockey_page(r.text)`
3. Collect all row dicts into a single list called `all_rows`
4. Add `time.sleep(1)` between requests (be polite to the server!)
5. Convert `all_rows` to a DataFrame called `hockey_stats`
6. Find the team with the most total Wins across all years

---

**💡 Hint 1 — URL pattern:**  
`f"https://www.scrapethissite.com/pages/forms/?page_num={page}"` where `page` goes from 1 to 24.

**💡 Hint 2 — Collecting results:**  
Use `all_rows = []`, then `all_rows.extend(parse_hockey_page(r.text))` inside the loop. `.extend()` adds multiple items at once (unlike `.append()` which would add a list-of-dicts as a single item).

**💡 Hint 3 — Finding the team with most wins:**  
After converting to a DataFrame, cast the `Wins` column to numeric with `pd.to_numeric(hockey_stats['Wins'], errors='coerce')`, then use `.groupby('Team')['Wins'].sum().idxmax()`.

In [50]:
# Your code here
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time

all_rows = []
for page in range(1, 25):
    r = requests.get(f"https://www.scrapethissite.com/pages/forms/?page_num={page}") 
    all_rows.extend(parse_hockey_page(r.text))
    time.sleep(1)

hockey_stats = pd.DataFrame(all_rows)
hockey_stats


,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Boston Bruins,1990,44,24,,0.55,299,264,35
1,Buffalo Sabres,1990,31,30,,0.388,292,278,14
2,Calgary Flames,1990,46,26,,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,,0.425,273,298,-25
...,...,...,...,...,...,...,...,...,...
577,Tampa Bay Lightning,2011,38,36,8,0.463,235,281,-46
578,Toronto Maple Leafs,2011,35,37,10,0.427,231,264,-33
579,Vancouver Canucks,2011,51,22,9,0.622,249,198,51
580,Washington Capitals,2011,42,32,8,0.512,222,230,-8


In [51]:
hockey_stats['Wins'] = pd.to_numeric(hockey_stats['Wins'], errors='coerce')
best_team = hockey_stats.groupby('Team Name')['Wins'].sum().idxmax()
print(best_team)

Detroit Red Wings


---
## Wrap-Up: What You've Learned

Congratulations — you've extracted real data using three different techniques! Here's how they compare:

### When to Use Which Tool

| | REST API | GraphQL | Web Scraping |
|--|----------|---------|--------------|
| **Best for** | Standard data access from any service | When you need precise, flexible queries | When no API exists |
| **Authentication** | Usually required (Bearer token, API key) | Usually required | Usually not required |
| **Data format** | JSON (fixed shape) | JSON (shape you define) | HTML (requires parsing) |
| **Reliability** | High — API contracts are stable | High — schema-driven | Low — HTML can change anytime |
| **Rate limits** | Common — check docs | Common — check docs | Unofficial — be conservative |
| **Pagination** | `page` + `per_page` params | `first`/`after` cursor | URL pattern (`?page=N`) |
| **Legal/ethical risk** | Low | Low | Check ToS + robots.txt |

**Rule of thumb:**
1. Check if there's a REST API first
2. If REST over-fetches and a GraphQL API exists, use that
3. Fall back to scraping only when no API is available

### What's Next

You've covered the fundamentals. Here are the natural next steps:

- **Rate limiting & retries** — production pipelines use `tenacity` or custom retry logic to handle `429 Too Many Requests` errors gracefully
- **Async requests** — `aiohttp` + `asyncio` lets you make hundreds of requests concurrently instead of one at a time
- **Dynamic pages** — some websites load data via JavaScript. `Selenium` or `Playwright` can control a real browser to scrape these
- **Scheduled pipelines** — combine what you've learned with `Apache Airflow` or `Prefect` to run extraction jobs on a schedule

Good luck! 🚀